<a href="https://colab.research.google.com/github/vaishnavipoojary2202/malicious-url-detection/blob/vaishnavi-feature-engineering/notebooks/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering

This notebook prepares URL data for malicious URL classification.

The feature engineering pipeline includes:

1. URL preprocessing
2. URL-based structural feature extraction
3. Character-level TF-IDF feature extraction
4. Feature combination
5. Train-test splitting
6. Preparation of model-ready data

In [ ]:
import pandas as pd
import numpy as np

import re
import string
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

from scipy.sparse import hstack, csr_matrix

import joblib
import os

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "ML mini" in dirs:
        matches.append(os.path.join(root, "ML mini"))

print("Found folders:")
for path in matches:
    print(path)

Found folders:
/content/drive/MyDrive/ML mini


In [8]:
print(os.listdir("/content/drive/MyDrive/ML mini"))

['dataset', 'Research papers']


In [9]:
print(os.listdir("/content/drive/MyDrive/ML mini/dataset"))

['dataset_phishing.csv']


In [10]:
dataset_path = "/content/drive/MyDrive/ML mini/dataset/dataset_phishing.csv"

In [11]:
print(os.path.exists(dataset_path))

True


In [12]:
import pandas as pd

df = pd.read_csv(dataset_path)

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (11430, 89)


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


In [13]:
print(df['status'].value_counts())

status
legitimate    5715
phishing      5715
Name: count, dtype: int64


In [14]:
data = df[['url', 'status']].copy()

print("Working dataset shape:", data.shape)
display(data.head())

Working dataset shape: (11430, 2)


,url,status
0,http://www.crestonwood.com/router.php,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,phishing
2,https://support-appleld.com.secureupdate.duila...,phishing
3,http://rgipt.ac.in,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,legitimate


legitimate → 0
phishing   → 1

In [15]:
data['label'] = data['status'].map({
    'legitimate': 0,
    'phishing': 1
})

print(data['label'].value_counts())

label
0    5715
1    5715
Name: count, dtype: int64


In [16]:
data = data.drop(columns=['status'])

display(data.head())

,url,label
0,http://www.crestonwood.com/router.php,0
1,http://shadetreetechnology.com/V4/validation/a...,1
2,https://support-appleld.com.secureupdate.duila...,1
3,http://rgipt.ac.in,0
4,http://www.iracing.com/tracks/gateway-motorspo...,0


In [17]:
data['url'] = data['url'].astype(str).str.strip()

print("Missing URLs:", data['url'].isna().sum())
print("Duplicate URLs:", data['url'].duplicated().sum())

Missing URLs: 0
Duplicate URLs: 1


In [18]:
data = data.drop_duplicates(subset=['url']).reset_index(drop=True)

print("Dataset shape after removing duplicate URLs:", data.shape)
print("Duplicate URLs:", data['url'].duplicated().sum())

Dataset shape after removing duplicate URLs: (11429, 2)
Duplicate URLs: 0


In [19]:
import re
from urllib.parse import urlparse

def extract_url_features(url):
    features = {}

    url = str(url)

    # Basic URL characteristics
    features['url_length'] = len(url)
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_letters'] = sum(c.isalpha() for c in url)

    # Special characters
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_at'] = url.count('@')
    features['num_question_marks'] = url.count('?')
    features['num_equals'] = url.count('=')
    features['num_ampersands'] = url.count('&')
    features['num_slashes'] = url.count('/')
    features['num_percent'] = url.count('%')
    features['num_underscores'] = url.count('_')
    features['num_colons'] = url.count(':')

    # HTTPS
    features['has_https'] = int(
        url.lower().startswith('https://')
    )

    # IP address detection
    features['has_ip'] = int(
        bool(
            re.search(
                r'(?:(?:25[0-5]|2[0-4][0-9]|1?[0-9]{1,2})\.){3}'
                r'(?:25[0-5]|2[0-4][0-9]|1?[0-9]{1,2})',
                url
            )
        )
    )

    # Hostname and path features
    try:
        parsed = urlparse(url)

        hostname = parsed.hostname or ''
        path = parsed.path or ''

        features['hostname_length'] = len(hostname)
        features['path_length'] = len(path)

        if hostname:
            parts = hostname.split('.')
            features['num_subdomains'] = max(len(parts) - 2, 0)
        else:
            features['num_subdomains'] = 0

    except Exception:
        features['hostname_length'] = 0
        features['path_length'] = 0
        features['num_subdomains'] = 0

    # Suspicious keywords
    suspicious_words = [
        'login',
        'verify',
        'verification',
        'secure',
        'account',
        'update',
        'confirm',
        'password',
        'signin',
        'bank',
        'payment'
    ]

    url_lower = url.lower()

    features['suspicious_keyword_count'] = sum(
        word in url_lower
        for word in suspicious_words
    )

    # URL shortening services
    shortening_domains = [
        'bit.ly',
        'tinyurl.com',
        'goo.gl',
        't.co',
        'ow.ly',
        'is.gd',
        'buff.ly'
    ]

    features['has_shortening_service'] = int(
        any(
            domain in url_lower
            for domain in shortening_domains
        )
    )

    return features

In [20]:
structural_features = data['url'].apply(extract_url_features)

structural_df = pd.DataFrame(
    structural_features.tolist()
)

display(structural_df.head())

print("Number of structural features:", structural_df.shape[1])
print("Structural feature shape:", structural_df.shape)

,url_length,num_digits,num_letters,num_dots,num_hyphens,num_at,num_question_marks,num_equals,num_ampersands,num_slashes,num_percent,num_underscores,num_colons,has_https,has_ip,hostname_length,path_length,num_subdomains,suspicious_keyword_count,has_shortening_service
0,37,0,30,3,0,0,0,0,0,3,0,0,1,0,0,19,11,1,0,0
1,77,17,53,1,0,0,0,0,0,5,0,0,1,0,0,23,47,0,0,0
2,126,19,88,4,1,0,1,3,2,5,0,2,1,1,0,50,20,3,2,0
3,18,0,13,2,0,0,0,0,0,2,0,0,1,0,0,11,0,1,0,0
4,55,0,45,2,2,0,0,0,0,5,0,0,1,0,0,15,33,1,0,0


Number of structural features: 20
Structural feature shape: (11429, 20)


In [21]:
print("Missing values:")
print(structural_df.isnull().sum())

print("\nTotal missing values:",
      structural_df.isnull().sum().sum())

Missing values:
url_length                  0
num_digits                  0
num_letters                 0
num_dots                    0
num_hyphens                 0
num_at                      0
num_question_marks          0
num_equals                  0
num_ampersands              0
num_slashes                 0
num_percent                 0
num_underscores             0
num_colons                  0
has_https                   0
has_ip                      0
hostname_length             0
path_length                 0
num_subdomains              0
suspicious_keyword_count    0
has_shortening_service      0
dtype: int64

Total missing values: 0


In [22]:
display(structural_df.describe().T)

,count,mean,std,min,25%,50%,75%,max
url_length,11429.0,61.120658,55.294849,12.0,33.0,47.0,71.0,1641.0
num_digits,11429.0,5.452271,16.320612,0.0,0.0,0.0,5.0,679.0
num_letters,11429.0,45.695861,38.811774,4.0,25.0,37.0,54.0,1301.0
num_dots,11429.0,2.480532,1.369671,1.0,2.0,2.0,3.0,24.0
num_hyphens,11429.0,0.997638,2.087157,0.0,0.0,0.0,1.0,43.0
num_at,11429.0,0.022224,0.155507,0.0,0.0,0.0,0.0,4.0
num_question_marks,11429.0,0.141220,0.364469,0.0,0.0,0.0,0.0,3.0
num_equals,11429.0,0.293202,0.998357,0.0,0.0,0.0,0.0,19.0
num_ampersands,11429.0,0.162306,0.821372,0.0,0.0,0.0,0.0,19.0
num_slashes,11429.0,4.289439,1.882266,2.0,3.0,4.0,5.0,33.0


In [23]:
from sklearn.model_selection import train_test_split

X_train_url, X_test_url, X_train_struct, X_test_struct, y_train, y_test = train_test_split(
    data['url'],
    structural_df,
    data['label'],
    test_size=0.20,
    random_state=42,
    stratify=data['label']
)

print("Training URL samples:", len(X_train_url))
print("Testing URL samples:", len(X_test_url))

print("Training structural features:", X_train_struct.shape)
print("Testing structural features:", X_test_struct.shape)

print("Training labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training URL samples: 9143
Testing URL samples: 2286
Training structural features: (9143, 20)
Testing structural features: (2286, 20)
Training labels:
label
0    4572
1    4571
Name: count, dtype: int64

Testing labels:
label
1    1143
0    1143
Name: count, dtype: int64


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    min_df=2,
    max_features=50000
)

X_train_tfidf = tfidf.fit_transform(X_train_url)
X_test_tfidf = tfidf.transform(X_test_url)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (9143, 50000)
Testing TF-IDF shape: (2286, 50000)


In [25]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix

scaler = StandardScaler()

X_train_struct_scaled = scaler.fit_transform(
    X_train_struct
)

X_test_struct_scaled = scaler.transform(
    X_test_struct
)

# Convert to sparse matrices
X_train_struct_scaled = csr_matrix(
    X_train_struct_scaled
)

X_test_struct_scaled = csr_matrix(
    X_test_struct_scaled
)

print("Training:", X_train_struct_scaled.shape)
print("Testing:", X_test_struct_scaled.shape)

Training: (9143, 20)
Testing: (2286, 20)


In [26]:
from scipy.sparse import hstack

X_train = hstack([
    X_train_tfidf,
    X_train_struct_scaled
]).tocsr()

X_test = hstack([
    X_test_tfidf,
    X_test_struct_scaled
]).tocsr()

print("Final training matrix:", X_train.shape)
print("Final testing matrix:", X_test.shape)

Final training matrix: (9143, 50020)
Final testing matrix: (2286, 50020)


In [27]:
import os
import joblib
from scipy.sparse import save_npz

processed_path = "/content/drive/MyDrive/ML mini/processed"

os.makedirs(processed_path, exist_ok=True)

# Save feature matrices
save_npz(
    f"{processed_path}/X_train.npz",
    X_train
)

save_npz(
    f"{processed_path}/X_test.npz",
    X_test
)

# Save labels
y_train.to_csv(
    f"{processed_path}/y_train.csv",
    index=False
)

y_test.to_csv(
    f"{processed_path}/y_test.csv",
    index=False
)

# Save fitted TF-IDF vectorizer
joblib.dump(
    tfidf,
    f"{processed_path}/tfidf_vectorizer.pkl"
)

# Save fitted scaler
joblib.dump(
    scaler,
    f"{processed_path}/structural_scaler.pkl"
)

# Save structural feature names
pd.Series(
    structural_df.columns
).to_csv(
    f"{processed_path}/structural_feature_names.csv",
    index=False
)

print("All processed files saved successfully!")
print("Location:", processed_path)

All processed files saved successfully!
Location: /content/drive/MyDrive/ML mini/processed


In [28]:
import os

print("Files in processed folder:\n")

for file in sorted(os.listdir(processed_path)):
    print("✓", file)

Files in processed folder:

✓ X_test.npz
✓ X_train.npz
✓ structural_feature_names.csv
✓ structural_scaler.pkl
✓ tfidf_vectorizer.pkl
✓ y_test.csv
✓ y_train.csv


## Feature Engineering Summary

The URL dataset was prepared for machine learning by:

1. Removing duplicate URLs.
2. Encoding the target variable:
   - Legitimate = 0
   - Phishing = 1
3. Extracting 20 structural URL features.
4. Splitting the dataset into training and testing sets using stratified sampling.
5. Extracting character-level TF-IDF features using 3–5 character n-grams.
6. Scaling the 20 structural features using StandardScaler.
7. Combining TF-IDF and structural features into a final sparse feature matrix.

### Final Feature Matrix

- Training samples: 9,143
- Testing samples: 2,286
- TF-IDF features: 50,000
- Structural features: 20
- Total features: 50,020

### Saved Artifacts

The processed feature matrices and preprocessing objects were saved to the shared Google Drive:

`ML mini/processed/`

These files will be used for model training and later deployment.
